# SumIterative, RecursiveSum and RecursiveSumLinear (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/SumIterative_RecursiveSum_RecursiveSumLinear_cpp.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `SumIterative`, `RecursiveSum` and `RecursiveSumLinear`: the algorithms in C++, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/SumIterative_RecursiveSum_RecursiveSumLinear_python.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/SumIterative_RecursiveSum_RecursiveSumLinear_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_cpp.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Each C++ program is written to a file by a `%%writefile` cell, then compiled with `g++` and run in the next cell: everything happens inside Colab.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `RecursiveSum` (PascalCase).

## The sum of an array: SumIterative, RecursiveSum, RecursiveSumLinear

- **SumIterative**$(a,n)$: a loop that accumulates the partial sums. $S(n)=3n+4$, extra memory $\Theta(1)$.
- **RecursiveSum**$(a,l,r)$: divide and conquer on the two halves (base case $l=r$). $2n-1$ calls, $S(n)=7n-5$, depth $\lceil\log_2 n\rceil+1$.
- **RecursiveSumLinear**$(a,n)$: decrease by one: $S(n)=3n-1$, depth $n$.

*Input*: an array of $n\ge1$ integers (for `RecursiveSum`, two indices $l\le r$). *Output*: the sum of the elements. The counters `steps` (lines of pseudo-code executed), `calls` and `maxDepth` (simultaneously active calls) measure the quantities analysed in the chapter; in `RecursiveSum` and `RecursiveSumLinear` there is a single `return` so that the depth can be updated.

On the array of the example $a=[9,8,12,3,5,14,6]$ ($n=7$) the sum is $57$; `RecursiveSum` makes $13$ calls and $44$ steps, with depth $4$. Then we run the trace of the table of the chapter on $a=[9,8,12,3,5]$: the column *stack* reports the active calls.

In [1]:
%%writefile sums.cpp
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
#include <cmath>
#include <cstdio>
#include <chrono>
#include <climits>
#include <functional>
#include <array>
using namespace std;

long long steps = 0;   // steps executed (lines of pseudo-code)
int calls = 0;         // calls executed
int depth = 0;         // calls active right now
int maxDepth = 0;      // maximum depth of the recursion

void Reset()
{
    steps = 0;
    calls = 0;
    depth = 0;
    maxDepth = 0;
}

string Fmt(const int *a, int n)
{
    string s = "[";
    for (int i = 0; i < n; i++)
    {
        if (i > 0) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

// Iterative sum of a[0..n-1]
int SumIterative(int n, int *a)
{
    int s = 0;
    int i = 0;
    steps += 2;                 // lines 1-2: s := 0, i := 1
    while (i < n)
    {
        steps += 3;             // true test + two lines of the body
        s = s + a[i];
        i = i + 1;
    }
    steps += 2;                 // false test + return
    return s;
}

// Sum of a[l..r] (0-based indices), l <= r, by divide and conquer
int RecursiveSum(int *a, int l, int r)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // line 1: if l = r
    int result;
    if (l == r)
    {
        steps++;                // line 2: return a[l] (base case)
        result = a[l];
    }
    else
    {
        steps += 4;             // lines 3-6: m, two calls, return
        int m = l + (r - l) / 2;                // divide
        int s1 = RecursiveSum(a, l, m);         // conquer (left half)
        int s2 = RecursiveSum(a, m + 1, r);     // conquer (right half)
        result = s1 + s2;                       // combine
    }
    depth--;
    return result;
}

// Sum of a[0..n-1] by decrease by one, n >= 1
int RecursiveSumLinear(int *a, int n)
{
    calls++;
    depth++;
    if (depth > maxDepth) maxDepth = depth;
    steps++;                    // line 1: if n = 1
    int result;
    if (n == 1)
    {
        steps++;                // line 2: return a[1]
        result = a[0];
    }
    else
    {
        steps += 2;             // lines 3-4: call and return
        int s = RecursiveSumLinear(a, n - 1);
        result = s + a[n - 1];
    }
    depth--;
    return result;
}

// ---- trace of RecursiveSum ----
int row = 0;
int maxStack = 0;

string Join(const vector<string> &v)
{
    string s;
    for (size_t i = 0; i < v.size(); i++)
    {
        if (i > 0) s += ",";
        s += v[i];
    }
    return s;
}

void Emit(const vector<string> &stack, const string &action)
{
    row++;
    if ((int)stack.size() > maxStack) maxStack = stack.size();
    printf("%2d | %s | %s\n", row, Join(stack).c_str(), action.c_str());
}

string P(int l, int r) { return "(" + to_string(l) + "," + to_string(r) + ")"; }

int RecursiveSumTrace(int *a, int l, int r, vector<string> &stack)
{
    stack.push_back(P(l, r));
    if (l == r)
    {
        Emit(stack, "l=r => return a[" + to_string(l) + "]=" + to_string(a[l]));
        stack.pop_back();
        return a[l];
    }
    int m = l + (r - l) / 2;
    Emit(stack, "l!=r, m=" + to_string(l) + "+(" + to_string(r) + "-" + to_string(l) + ")/2=" + to_string(m) + "; call " + P(l, m));
    int s1 = RecursiveSumTrace(a, l, m, stack);
    Emit(stack, "s1=" + to_string(s1) + "; call " + P(m + 1, r));
    int s2 = RecursiveSumTrace(a, m + 1, r, stack);
    Emit(stack, "s2=" + to_string(s2) + "; return " + to_string(s1) + "+" + to_string(s2) + "=" + to_string(s1 + s2));
    stack.pop_back();
    return s1 + s2;
}

#ifndef NOMAIN
int main()
{
    int a[] = {9, 8, 12, 3, 5, 14, 6};
    int n = 7;
    printf("a = %s   n = %d\n", Fmt(a, n).c_str(), n);
    Reset();
    int s = SumIterative(n, a);
    printf("SumIterative       = %d   steps=%lld (3n+4=%d)\n", s, steps, 3 * n + 4);
    Reset();
    s = RecursiveSum(a, 0, n - 1);
    printf("RecursiveSum       = %d   calls=%d (2n-1=%d)  steps=%lld (7n-5=%d)  maxDepth=%d\n", s, calls, 2 * n - 1, steps, 7 * n - 5, maxDepth);
    Reset();
    s = RecursiveSumLinear(a, n);
    printf("RecursiveSumLinear = %d   calls=%d  steps=%lld (3n-1=%d)  maxDepth=%d\n", s, calls, steps, 3 * n - 1, maxDepth);

    int b[] = {9, 8, 12, 3, 5};
    printf("\n");
    printf("a = %s   RecursiveSum(a, 0, 4)\n", Fmt(b, 5).c_str());
    printf(" # | stack | action\n");
    vector<string> stack;
    int res = RecursiveSumTrace(b, 0, 4, stack);
    printf("result %d, rows %d, maximum number of active calls %d\n", res, row, maxStack);
    return 0;
}
#endif

Writing sums.cpp


In [2]:
!g++ -std=c++17 -O2 -o sums sums.cpp && ./sums

a = [9, 8, 12, 3, 5, 14, 6]   n = 7
SumIterative       = 57   steps=25 (3n+4=25)
RecursiveSum       = 57   calls=13 (2n-1=13)  steps=44 (7n-5=44)  maxDepth=4
RecursiveSumLinear = 57   calls=7  steps=20 (3n-1=20)  maxDepth=7

a = [9, 8, 12, 3, 5]   RecursiveSum(a, 0, 4)
 # | stack | action
 1 | (0,4) | l!=r, m=0+(4-0)/2=2; call (0,2)
 2 | (0,4),(0,2) | l!=r, m=0+(2-0)/2=1; call (0,1)
 3 | (0,4),(0,2),(0,1) | l!=r, m=0+(1-0)/2=0; call (0,0)
 4 | (0,4),(0,2),(0,1),(0,0) | l=r => return a[0]=9
 5 | (0,4),(0,2),(0,1) | s1=9; call (1,1)
 6 | (0,4),(0,2),(0,1),(1,1) | l=r => return a[1]=8
 7 | (0,4),(0,2),(0,1) | s2=8; return 9+8=17
 8 | (0,4),(0,2) | s1=17; call (2,2)
 9 | (0,4),(0,2),(2,2) | l=r => return a[2]=12
10 | (0,4),(0,2) | s2=12; return 17+12=29
11 | (0,4) | s1=29; call (3,4)
12 | (0,4),(3,4) | l!=r, m=3+(4-3)/2=3; call (3,3)
13 | (0,4),(3,4),(3,3) | l=r => return a[3]=3
14 | (0,4),(3,4) | s1=3; call (4,4)
15 | (0,4),(3,4),(4,4) | l=r => return a[4]=5
16 | (0,4),(3,4) | s2=5; retur

**Try it yourself.** Change the array `a` (also its length) and compare steps, calls and depth: only `RecursiveSumLinear` has depth $n$.

In [3]:
%%writefile prova_sums.cpp
#define NOMAIN
#include "sums.cpp"

int main()
{
    // Change the array: compare steps, calls and depth of the three algorithms.
    int a[] = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16};
    int n = 16;
    Reset(); int s = SumIterative(n, a);
    printf("%d %lld\n", s, steps);
    Reset(); s = RecursiveSum(a, 0, n - 1);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    Reset(); s = RecursiveSumLinear(a, n);
    printf("%d %lld %d %d\n", s, steps, calls, maxDepth);
    return 0;
}

Writing prova_sums.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_sums prova_sums.cpp && ./prova_sums

136 52
136 107 31 5
136 47 16 16


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).